*第二部分：高级学习算法*

一、工作原理

$词嵌入（Word Embeddings） $的本质：将语义相似的词放在空间中相互靠近的位置。

1. 多分类任务和softmax激活函数

softmax的本质：将数字转变为概率(所有输出都在[0, 1]范围内，所有输出和为1)

非线性激活函数：$ReLU$

如何衡量猜测的准确性：

上一节的回归任务：$MSE(均方误差)$， 衡量“数字离得有多远”
在分类任务中：$交叉熵损失(Cross-Enttopy Loss)$, 衡量“概率分布有多接近”

这一节使用的交叉熵，就是在模型对正确答案给出的概率太低的话，损失值会迅速上升，从而强迫模型下一次给正确答案更高的分数/概率。
对应 $torch.nn.CrossEntropyLoss()$


2. 序列处理的演变（ 高级学习算法）：

- CNN(卷积神经网络)：通过“滑动窗口”关注空间上的局部特征（如图片里的形状）。

- RNN(循环神经网络)：通过“记忆传递”从前往后传递记忆，关注时间上的先后顺序（如句子里的前因后果）。

- Self-Attention(自注意力机制)：通过“注意力”让模型在处理每一个词时，都能自动去寻找整个句子里和它最相关的其他词，而不管它们离得有多远。（Attention is all you need）

Transformer处理数据的三部曲：

- 查询（Query）：我想找什么？（比如：“它”在找它指代的对象）

- 键（Key）：我有什么特征？（比如：“猫”这个词带有“生物”、“名词”的标签）

- 值（Value）：我包含了什么具体信息？（比如：“猫”这个词代表的具体语义）

- 简称：Q, K, V机制。

直观理解：
- 在 Transformer 的自注意力机制中，系统会计算 Query（借书单） 与每一个 Key（书脊标签） 的相似度。相似度越高，模型就会分配越高的权重，从而从对应的 Value（书本内容） 中提取越多的信息。

数学直觉：
- 匹配（Matching）：
$Query \cdot Key$。如果两个向量的方向一致（相似），结果就会很大。
- 归一化（Softmax）：模型会把所有的匹配分数丢进 Softmax，转化成一组总和为 $1$ 的权重（比如：词 A 占 $80\%$，词 B 占 $20\%$）。
- 提取（Extraction）：用这组权重去加权求和所有的 Value(就是三部曲中的V)从而得到当前向量的最终表示，如：
  - $\text{最终表示} = (0.4 \times \text{Value}_{\text{苹果}}) + (0.5 \times \text{Value}_{\text{手机}}) + (0.1 \times \text{Value}_{\text{很好用}})$
  - 由于“手机”拿走了 50% 的权重，这个“苹果”的最终表示里会注入大量的“科技、电子产品”的信息。

虽然它原本的词嵌入可能兼具“水果”和“品牌”的意思，但经过这次提取，它的含义被锁定在了“品牌/电子产品”上。

- 由此我们可以将"注意力"集中在最相关的词语上。

*梳理*

Transformer核心逻辑 ：
- 词嵌入 ： 文字变成空间内的坐标(向量)
- Softmax: 把得分变为概率分配，体现相对大小
- Q,K,V机制：根据相关性得到最终表示。


二、Transformer架构的三大支柱

1. 编码器与解码器

- 编码器(Encoder)：负责"理解"，把输入的整句话读一遍，利用自注意力机制，弄清楚每个词之间的关系。
   - 产出：一堆包含上下文信息的特征向量。
   - 特点：一次性看整句，全面

- 解码器(Decoder)：负责"表达", 根据给出的特征向量，一个一个产生翻译结果。
   - 特点：不仅看编码器给出了什么信息，还要看刚才已经写了什么。


- 关键细节：Masking(掩盖)机制

   - 在 Decoder 生成翻译时，它不能“偷看”未来的答案。比如翻译到 "eat" 的时候，它不能提前知道后面要说 "apple"。

   - 为了训练这个能力，模型会使用一种叫 Masked Self-Attention（带掩码的自注意力） 的技术。
   - 掩盖机制的本质是：虽然我们为了计算效率把全句都给了模型，但通过数学手段（把未来的词权重设为 0），我们人为地让模型在预测第 $n$ 个词时，只能“看见”前 $n-1$ 个词。

2. 多头注意力($Multi-Head Attention$)
   
- "多头": 对句子的理解可以从更多方面，将不同方面的信息拼接在一起。如：

    - “那只猫横穿了马路。”

    - 如果模型只有一个“头”，它可能只能注意到最明显的联系（比如“猫”和“横穿”的动作联系）。但如果有多个“头”，它们可以分工协作：

    - 第一个头：关注语法结构（“猫”是主语，“横穿”是谓语）。

    - 第二个头：关注属性描述（“那只”修饰“猫”）。

    - 第三个头：关注空间关系（“横穿”和“马路”）。

    - 最后，模型会把这些不同视角提取到的信息“拼接”在一起。这让模型对句子的理解不再是单一的，而是立体的。

3. 残差连接与归一化($Residual \space \& \space LayerNorm$)

- 梯度消失现象：在深度学习中，随着层数变深（Transformer 通常有几十层甚至上百层），信息在传递过程中会逐渐衰减或扭曲，这被称为梯度消失。
  - 本质：神经网络的学习依赖反向传播，底层机理是计算损失函数对前面层参数的倒数，使用链式法则。若每一层的贡献(梯度)都小于一，连乘几次之后就会接近0，最后靠近输入层(因为backward是从最后一层开始的)的层几乎收不到有效的更新信号
- 因此引入两道保险：
    - 残差连接（Residual Connection）：它在每一层旁边修了一条“快车道”。原始信息可以直接跳过这一层的复杂运算，直接传给下一层。公式直觉：$\text{输出} = \text{原始输入} + \text{这一层的学习结果}$。这样即使这一层没学好，原始信息也不会丢。**可以理解为 $output = f(x) + x$**
    - 层归一化（Layer Normalization）：它像是一个“恒温器”。每一层运算完后，数据的大小波动可能很大，归一化会把数据重新调整到一个稳定的范围（均值为 0，方差为 1），防止模型计算数值爆炸。

拓展：
GPT(Generative Pre-trained Transformer) 

- 本质上是一个只有解码器(Decoder-only)的架构，它只需要根据前文(prompt)生成后文。

- 即：1. 只需理解已经说过的词 2. 预测概率最大的下一个词。 

微调(Fine-tuning)

实践：微缩版GPT: 学习特定风格的文本。

代码流程：
1. ln1 层归一化，准备好干净的数据输入注意力机制
2. sa， 寻找词与词之间的联系
3. x + sa 残差加法
4. ln2 层归一化，为接下来的逻辑转换做准备
5. ffwd 前馈运算：对每个词的特征进行深度加工
6. x + ffwd 残差加法，最终整合，输出给下一个Block

这里的前馈网络的工作原理有点像：

```/py
    self.ffwd = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd),
    )
```

In [ ]:
# 核心骨架

import torch
import torch.nn as nn
import torch.nn.functional as F

class Block(nn.Module):
    '''一个完整的Transformer块'''
    def __init__(self, n_embd, n_head):
        '''
        n_embd: 词嵌入维度(Embedding Dimension)：每个词被转化成的向量长度
        n_head: 多头注意力的头数
        n_embd // n_head: 每个头的维度

        ln1, ln2: Layer Normalization 层归一化
        sa : Self-Attention 自注意力
        ffwd : Feed-Forward Network(有时也写FFN)前馈网络
        '''
        super().__init__()

        # 1. 层归一化
        self.ln1 = nn.LayerNorm(n_embd)
        # 2. 多头注意力(命名为雷达sa)， 通过Q/K/V, 根据与其他词的相关性重新计算自己的表达
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        # 3. 前馈网络(消化)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

        # 多头注意力之前和前馈网络前都有一个LayerNorm, 
    
    def forward(self, x):
        # 残差连接： output = x + f(x)
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x
    
class NanoGPT(nn.Module):
    def forward(self, idx, targets=None):
        '''
        idx: 输入字符的ID序列
        target: 
        '''

        # 1. 加上位置编码，让模型了解顺序
        x = self.token_embedding_table(idx) + self.position_embedding_table

        # 2. 经过多层Transformer 块
        x = self.blocks(x)

        # 3. 归一化并输出概率
        logits = self.lm_head(self.ln1(x))

        # LM Head (Language Model Head): 这是一个线性层，它的作用是把隐藏层的向量投影到词表大小的空间。
        # Logits: 它的输出是一个分数值（Logits）。比如你的词表有 5000 个字，它就会输出 5000 个数。哪个数字最大，就代表模型认为下一个字最可能是谁。

        return logits

1. 数据处理与Tokenization
- 1. 建立词表：先找出文本中出现过的所有字符
- 2. 映射表：两个工具：stoi(字符转数字)， itos(数字转字符)
- 3. 编码：将整篇文章转成一个很长的数字列表(Tensor)

    - 举例：假设我们的文本只有一句话：“周杰伦唱双截棍”。

    - 词表：['伦', '双', '周', '唱', '杰', '棍', '截']
    - 映射：'周' -> 2, '杰' -> 4, '伦' -> 0 ...
    - 编码后的数据：[2, 4, 0, 3, 1, 6, 5]

In [ ]:
# 随机抓取一些片段喂给模型(Tokenization)：

def get_batch(data, block_size, batch_size):
    # 随机挑选batch_size个起始位置，
    # len(data)要减去bloack_size是为了防止越界，(batch_size,)是输出张量的形状。
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # 根据起始位置截取输入x
    x = torch.stack([data[i : i + block_size] for i in ix])

    # 截取对应y(x后一位， 作为训练集的答案)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])

    return x, y

# 这里使用到了torch.stack
# stack : 意为堆叠, 将一个包含多个张量的列表在一个新维度堆叠起来，变成一个高维的张量矩阵
# 如，上面使用列表推导式产生的是装有几个张量的Python列表，而神经网络只认识[batch_size, block_size]的张量
# 这时就可以使用torch.stack将这几个张量堆叠成一个张量矩阵。

2. 自注意力机制: Q/K/V
- 1. 产生分身: 将输入x通过不同的线性层(nn.Linear), 把输入的x变成Q, K, V三个矩阵
- 2. 打分: 计算 $Q \cdot K^T$, 得到注意力矩阵，再对其进行归一化(Softmax)，变成了一个权重矩阵，所有权重加起来等于1，每个权重都代表一个词对另一个词的注意力(概率)
- 3. 加权融合: 将权重矩阵与V(Value, 输入x所包含的内容)相乘, 获得最终 $ Output = Attention \times V $, 代表每个词对某个特征的信息偏向。如"apple" 包括 60% "sweet"的特征信息。 
- (拓展: 计算权重矩阵时会除以特征维度(即每个特征向量的长度)的平方根，避免某个词权重过大导致忽略了其他词语)

3. 多头注意力机制
- 假如有512个特征维度，如果只用一个"头", 会导致顾此失彼，有些特征不能放在一起比较。如"味道"和"质量"。
- 因此采用多头注意力机制，若有8个头,则每个头注意: $ 512 \div 8 = 64 $个特征维度。

- 代码中的切分技巧:
  - 假设输入的x : (batch_size, block_size, 512(n_embd, 每个词被转换成的向量长度))
  - 1. 线性变换: 先用nn.Linear(512, 512) 算出总的Q, K, V
  - 2. 变形: 把形状变成 (batch_size, block_size, 8 (即n_head), 64 (即n_embd // n_head, 每个头的特征维度)) 
  - 3. 转置: 把"头"的数量提到前面, 变成(batch_size, 8, block_size, 64)
  
  - 思考: 为什么不直接变形，还要多此一举的转置一下？